# 14 · Стереозрение и 3D-реконструкция / Stereo vision and 3D reconstruction
**RU.** Реальная пара Middlebury 2014 Motorcycle: два ректифицированных RGB-изображения 500×741 и эталонная disparity с NaN/inf там, где GT отсутствует. Наблюдение — соответствующие пиксели двух камер; признаки — яркость и локальная текстура; цель — disparity, затем метрические координаты 3D. [Источник Middlebury](https://vision.middlebury.edu/stereo/data/scenes2014/), [официальная копия scikit-image](https://scikit-image.org/docs/stable/api/skimage.data.html#skimage.data.stereo_motorcycle). Middlebury разрешает использование и публикацию с цитированием Scharstein et al., GCPR 2014.
**EN.** Real Middlebury 2014 Motorcycle pair: two rectified 500×741 RGB images and reference disparity with nonfinite values for unavailable GT. Observations are corresponding camera pixels, features are intensity and local texture, targets are disparity and metric 3D coordinates. The scikit-image copy downsamples the original by four. Middlebury permits use/publication with citation of Scharstein et al., GCPR 2014.

**RU.** CPU, около 2.5 МБ загрузок, закреплённые SHA256. Единственная сцена, параметры SGBM заданы до оценки: это воспроизводимая демонстрация, а не test обобщения на новые сцены. Пиксели пространственно зависимы, случайное разбиение не создаёт независимый test. Для сравнения моделей нужны дополнительные целиком отложенные сцены. Истинная disparity нужна только оценщику.
**EN.** CPU, about 2.5 MB of downloads with pinned SHA256 hashes. One scene with SGBM settings fixed before evaluation: a reproducible demonstration, not evidence of generalization. Spatially dependent pixels do not become an independent test through random splitting. Model comparison requires additional held-out scenes. Ground-truth disparity is used only by evaluation.

**Калибровка / Calibration.** f=994.978 px, cx=311.193 px, cy=254.877 px, doffs=cx_right−cx_left=31.086 px, B=193.001 mm=0.193001 m. Эти значения относятся к уменьшенным изображениям. / These values apply to the downsampled images. Z=fB/(d+doffs), X=(u−cx)Z/f, Y=(v−cy)Z/f. Camera coordinates: X right, Y down, Z forward.


In [ ]:
from pathlib import Path  # RU: управляем локальным кэшем; EN: manage the local cache.
import urllib.request,hashlib,json  # RU: загружаем и проверяем данные; EN: download and verify data.
import numpy as np  # RU: вычисляем геометрию; EN: compute geometry.
import cv2  # RU: сопоставляем стереопару; EN: match the stereo pair.
import matplotlib.pyplot as plt  # RU: визуализируем измерения; EN: visualize measurements.
data=next((p for p in [Path('build/data'),Path('../build/data'),Path('data')] if (p/'motorcycle_left.png').exists()),Path('data'))  # RU: переиспользуем кэш; EN: reuse the cache.
data.mkdir(parents=True,exist_ok=True)  # RU: создаём каталог при необходимости; EN: create a directory if needed.
hashes={'motorcycle_left.png':'db18e9c4157617403c3537a6ba355dfeafe9a7eabb6b9b94cb33f6525dd49179','motorcycle_right.png':'5fc913ae870e42a4b662314bc904d1786bcad8e2f0b9b67dba5a229406357797','motorcycle_disp.npz':'2e49c8cebff3fa20359a0cc6880c82e1c03bbb106da81a177218281bc2f113d7'}  # RU: закрепляем официальный реестр scikit-image; EN: pin the official scikit-image registry.
for name,digest in hashes.items():  # RU: проверяем каждый ресурс; EN: verify each resource.
    path=data/name  # RU: задаём путь файла; EN: set the file path.
    if not path.exists(): urllib.request.urlretrieve('https://raw.githubusercontent.com/scikit-image/scikit-image/v0.25.2/skimage/data/'+name,path)  # RU: скачиваем версионированную копию; EN: download a versioned copy.
    assert hashlib.sha256(path.read_bytes()).hexdigest()==digest  # RU: отклоняем изменённые данные; EN: reject changed data.
left=cv2.cvtColor(cv2.imread(str(data/'motorcycle_left.png')),cv2.COLOR_BGR2RGB)  # RU: читаем левое RGB-изображение; EN: read the left RGB image.
right=cv2.cvtColor(cv2.imread(str(data/'motorcycle_right.png')),cv2.COLOR_BGR2RGB)  # RU: читаем правое RGB-изображение; EN: read the right RGB image.
gt=np.load(data/'motorcycle_disp.npz')['arr_0']  # RU: загружаем эталонные смещения; EN: load reference disparities.
calib={'f':994.978,'cx':311.193,'cy':254.877,'doffs':31.086,'baseline_m':.193001}  # RU: используем калибровку четверти разрешения; EN: use quarter-resolution calibration.
matcher=cv2.StereoSGBM_create(minDisparity=0,numDisparities=128,blockSize=5,P1=8*25,P2=32*25,disp12MaxDiff=1,uniquenessRatio=10,speckleWindowSize=50,speckleRange=2,mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)  # RU: фиксируем настройки до оценки; EN: fix settings before evaluation.
raw=matcher.compute(cv2.cvtColor(left,cv2.COLOR_RGB2GRAY),cv2.cvtColor(right,cv2.COLOR_RGB2GRAY))  # RU: вычисляем disparity в фиксированной точке; EN: compute fixed-point disparity.
assert left.shape==right.shape==(500,741,3) and gt.shape==(500,741)  # RU: проверяем согласование изображений и GT; EN: verify image and GT alignment.


## 1 · Формат disparity / Disparity representation
**RU · 5–10 минут.** `decode_disparity(raw,min_disparity)` возвращает (d,valid): d float32 = raw/16, valid = конечное d > min_disparity. Это консервативный договор занятия: нулевые смещения тоже исключаем. Не заменяйте карту цветной картинкой до вычислений. Покажите d и valid рядом с левой фотографией. Объясните, почему результат compute ещё не метры.
**EN · 5–10 minutes.** Return float32 d=raw/16 and a boolean mask for finite d>min_disparity. Our conservative teaching policy also excludes zero disparity. Keep the numeric map for calculations; do not replace it with a colour visualization. Display d and valid next to the left photograph. Explain why compute does not return metres.

In [ ]:
print(raw.dtype,raw.min(),raw.max())  # RU: изучаем диапазон фиксированной точки; EN: inspect fixed-point range.

In [ ]:
def decode_disparity(raw,min_disparity):  # RU: переводим результат OpenCV в пиксели; EN: convert OpenCV output to pixels.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
disp,valid=decode_disparity(raw,0)  # RU: применяем оговорённую политику; EN: apply the declared policy.
assert disp.dtype==np.float32 and valid.dtype==bool  # RU: проверяем контракт типов; EN: check the type contract.
assert np.allclose(disp,raw.astype(float)/16) and np.array_equal(valid,disp>0)  # RU: проверяем масштаб и фильтр; EN: verify scale and filtering.
fig,axs=plt.subplots(1,3,figsize=(15,4))  # RU: сравниваем вход и результат; EN: compare input and output.
for ax,img,title in zip(axs,[left,disp,valid],['Left','Disparity (px)','Valid']): ax.imshow(img);ax.set_title(title);ax.axis('off')  # RU: подписываем разные величины; EN: label distinct quantities.
plt.show()  # RU: выводим фотографии и измерения; EN: display photographs and measurements.


## 2 · Ошибка и покрытие / Error and coverage
**RU · 5–10 минут.** `disparity_metrics(pred,truth)` возвращает coverage,mae,bad2. GT-область: finite truth>0. Оценка допустима при finite pred>0. coverage — доля допустимых среди всех GT; mae — средняя абсолютная ошибка на пересечении; bad2 — доля всех GT, у которых предсказание отсутствует ИЛИ ошибка >2 px. Если GT нет, все метрики NaN; если нет пересечения, mae=NaN, coverage=0, bad2=1. Объясните, почему нельзя показывать только mae.
**EN · 5–10 minutes.** Return coverage,mae,bad2. Reference region is finite truth>0; predictions must be finite and positive. Coverage is valid predictions divided by all GT pixels. MAE uses their intersection. bad2 counts missing predictions OR errors >2 pixels over all GT. No GT gives all NaN; no intersection gives NaN MAE, zero coverage and bad2=1. Explain why MAE alone is insufficient.

In [ ]:
print('GT coverage:',np.mean(np.isfinite(gt)&(gt>0)))  # RU: измеряем доступность разметки; EN: measure annotation availability.

In [ ]:
def disparity_metrics(pred,truth):  # RU: оцениваем точность вместе с покрытием; EN: evaluate accuracy together with coverage.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
metric=disparity_metrics(disp,gt)  # RU: оцениваем фиксированный SGBM; EN: evaluate fixed SGBM.
perfect=disparity_metrics(gt,gt)  # RU: используем реальный GT как контроль; EN: use real GT as a control.
assert perfect['coverage']==1 and perfect['mae']==perfect['bad2']==0  # RU: проверяем идеальный результат; EN: check perfect predictions.
assert 0<=metric['coverage']<=1 and 0<=metric['bad2']<=1  # RU: проверяем диапазоны долей; EN: check fraction ranges.
print(metric)  # RU: показываем все метрики вместе; EN: report all metrics together.


## 3 · Метрическая глубина / Metric depth
**RU · 5–10 минут.** `depth_from_disparity(d,valid,f,B,doffs)` вернёт float-массив Z той же формы. На finite d, valid=True и d+doffs>0 используйте Z=f*B/(d+doffs), иначе NaN. B задан в метрах. f и B должны быть положительными, иначе ValueError. Постройте карту глубины и сравните с ошибочной формулой без doffs. Не делайте вывод об ошибке камеры по одной картинке.
**EN · 5–10 minutes.** Return a float depth array using Z=f*B/(d+doffs) for finite d, valid=True and positive denominator, otherwise NaN. B is in metres. Reject nonpositive f or B with ValueError. Visualize depth and compare with the incorrect formula omitting doffs. One image alone cannot diagnose camera calibration.

In [ ]:
print(calib)  # RU: проверяем единицы и сдвиг главных точек; EN: inspect units and principal-point offset.

In [ ]:
def depth_from_disparity(d,valid,f,B,doffs):  # RU: восстанавливаем глубину ректифицированной пары; EN: recover rectified stereo depth.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
depth=depth_from_disparity(disp,valid,calib['f'],calib['baseline_m'],calib['doffs'])  # RU: используем согласованную калибровку; EN: use consistent calibration.
assert np.allclose(depth[valid],calib['f']*calib['baseline_m']/(disp[valid]+calib['doffs']))  # RU: сверяем формулу; EN: verify the formula.
assert np.isnan(depth[~valid]).all()  # RU: проверяем пропуски; EN: verify missing values.
plt.imshow(depth,vmin=1,vmax=6);plt.colorbar(label='Z, metres');plt.show()  # RU: показываем глубину с единицами; EN: display depth with units.


## 4 · Облако точек / Point cloud
**RU · 5–10 минут.** `backproject(depth,f,cx,cy,step)` вернёт (points,pixels): точки (N,3) XYZ и индексы (N,2) (v,u). Берите только конечную положительную глубину в узлах u%step==v%step==0, порядок строковый. X=(u-cx)Z/f, Y=(v-cy)Z/f. step — положительное целое. Для step=8 окрасьте точки по left[v,u]. Спроецируйте обратно и проверьте восстановление u,v. Объясните различие облака и полной поверхности объекта.
**EN · 5–10 minutes.** Return (N,3) XYZ points and (N,2) pixel indices (v,u). Select finite positive depth on the step-spaced grid in row-major order. Use X=(u-cx)Z/f and Y=(v-cy)Z/f. Require positive integer step. Colour step=8 points using left[v,u], reproject to verify u,v, and explain why this cloud is not a complete object surface.

In [ ]:
print('Finite depth:',np.isfinite(depth).sum())  # RU: оцениваем размер облака до прореживания; EN: estimate cloud size before subsampling.

In [ ]:
def backproject(depth,f,cx,cy,step):  # RU: переводим пиксели в координаты камеры; EN: convert pixels to camera coordinates.
    raise NotImplementedError('TODO / Ваш код')  # RU: реализуйте контракт; EN: implement the contract.

In [ ]:
points,pixels=backproject(depth,calib['f'],calib['cx'],calib['cy'],8)  # RU: получаем разреженное облако; EN: obtain a sparse cloud.
assert np.allclose(calib['f']*points[:,0]/points[:,2]+calib['cx'],pixels[:,1])  # RU: проверяем обратную проекцию u; EN: verify u reprojection.
assert np.allclose(calib['f']*points[:,1]/points[:,2]+calib['cy'],pixels[:,0])  # RU: проверяем обратную проекцию v; EN: verify v reprojection.
fig=plt.figure(figsize=(10,6));ax=fig.add_subplot(111,projection='3d')  # RU: создаём научную 3D-визуализацию; EN: create a scientific 3D view.
ax.scatter(points[:,0],points[:,2],-points[:,1],c=left[pixels[:,0],pixels[:,1]]/255,s=2)  # RU: отображаем Y вверх только на графике; EN: flip Y only for display.
ax.set_xlabel('X m');ax.set_ylabel('Z m');ax.set_zlabel('-Y m');plt.show()  # RU: подписываем оси и единицы; EN: label axes and units.
